In [0]:
spark

In [0]:
df = spark.read.csv('/Volumes/workspace/default/raw_data/weather_data.csv', header=True, inferSchema=True)
display(df)


day,temperature,windspeed,event
1/1/2017,32.0,6.0,Rain
1/4/2017,null,9.0,Sunny
1/5/2017,28.0,null,Snow
1/6/2017,null,7.0,null
1/7/2017,32.0,null,Rain
1/8/2017,null,null,Sunny
1/9/2017,null,null,null
1/10/2017,34.1,8.1,Cloudy
1/11/2017,40.0,12.0,Sunny


In [0]:
from pyspark.sql import functions as F
new_df = df.filter(F.col("temperature").isNull())
new_df.show()

+--------+-----------+---------+-----+
|     day|temperature|windspeed|event|
+--------+-----------+---------+-----+
|1/4/2017|       NULL|      9.0|Sunny|
|1/6/2017|       NULL|      7.0| NULL|
|1/8/2017|       NULL|     NULL|Sunny|
|1/9/2017|       NULL|     NULL| NULL|
+--------+-----------+---------+-----+



In [0]:
df.filter(df.temperature.isNull()).count()

4

In [0]:
temp_nulls_sum = df.select(
    F.sum(F.col("temperature").isNull().cast("int")).alias("temperature"),
)
temp_nulls_sum.show()


+-----------+
|temperature|
+-----------+
|          4|
+-----------+



In [0]:
null_count = df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
)
null_count.show()

+---+-----------+---------+-----+
|day|temperature|windspeed|event|
+---+-----------+---------+-----+
|  0|          4|        4|    2|
+---+-----------+---------+-----+



In [0]:
drop_null = df.na.drop(how="any")
drop_null.show()

+---------+-----------+---------+------+
|      day|temperature|windspeed| event|
+---------+-----------+---------+------+
| 1/1/2017|       32.0|      6.0|  Rain|
|1/10/2017|       34.1|      8.1|Cloudy|
|1/11/2017|       40.0|     12.0| Sunny|
+---------+-----------+---------+------+



In [0]:
df.na.drop(how="any").show()
df.na.drop(how="all").show()
df.na.drop(how="all", subset=["temperature", "windspeed"]).show()


+---------+-----------+---------+------+
|      day|temperature|windspeed| event|
+---------+-----------+---------+------+
| 1/1/2017|       32.0|      6.0|  Rain|
|1/10/2017|       34.1|      8.1|Cloudy|
|1/11/2017|       40.0|     12.0| Sunny|
+---------+-----------+---------+------+

+---------+-----------+---------+------+
|      day|temperature|windspeed| event|
+---------+-----------+---------+------+
| 1/1/2017|       32.0|      6.0|  Rain|
| 1/4/2017|       NULL|      9.0| Sunny|
| 1/5/2017|       28.0|     NULL|  Snow|
| 1/6/2017|       NULL|      7.0|  NULL|
| 1/7/2017|       32.0|     NULL|  Rain|
| 1/8/2017|       NULL|     NULL| Sunny|
| 1/9/2017|       NULL|     NULL|  NULL|
|1/10/2017|       34.1|      8.1|Cloudy|
|1/11/2017|       40.0|     12.0| Sunny|
+---------+-----------+---------+------+

+---------+-----------+---------+------+
|      day|temperature|windspeed| event|
+---------+-----------+---------+------+
| 1/1/2017|       32.0|      6.0|  Rain|
| 1/4/2017|   

In [0]:
df.fillna(0).show()

+---------+-----------+---------+------+
|      day|temperature|windspeed| event|
+---------+-----------+---------+------+
| 1/1/2017|       32.0|      6.0|  Rain|
| 1/4/2017|        0.0|      9.0| Sunny|
| 1/5/2017|       28.0|      0.0|  Snow|
| 1/6/2017|        0.0|      7.0|  NULL|
| 1/7/2017|       32.0|      0.0|  Rain|
| 1/8/2017|        0.0|      0.0| Sunny|
| 1/9/2017|        0.0|      0.0|  NULL|
|1/10/2017|       34.1|      8.1|Cloudy|
|1/11/2017|       40.0|     12.0| Sunny|
+---------+-----------+---------+------+



In [0]:
df.fillna(
    {
        "temperature": 0,
        "windspeed": 10,
    }
).show()

+---------+-----------+---------+------+
|      day|temperature|windspeed| event|
+---------+-----------+---------+------+
| 1/1/2017|       32.0|      6.0|  Rain|
| 1/4/2017|        0.0|      9.0| Sunny|
| 1/5/2017|       28.0|     10.0|  Snow|
| 1/6/2017|        0.0|      7.0|  NULL|
| 1/7/2017|       32.0|     10.0|  Rain|
| 1/8/2017|        0.0|     10.0| Sunny|
| 1/9/2017|        0.0|     10.0|  NULL|
|1/10/2017|       34.1|      8.1|Cloudy|
|1/11/2017|       40.0|     12.0| Sunny|
+---------+-----------+---------+------+



In [0]:
avg_temp = df.select(F.avg(F.col("temperature"))).collect()[0][0]
print(avg_temp)

avg_temp_1 = df.select(F.mean(F.col("temperature")).alias("avg_temp"))
avg_temp_1.show()

avg_temp_3 = df.agg(F.mean("temperature")).first()[0]
avg_temp_3

# df.fillna(
#     {
#         "temperature": F.mean(df.temperature),
#         "windspeed": 10,
#     },
# ).show()

33.22
+--------+
|avg_temp|
+--------+
|   33.22|
+--------+



33.22

In [0]:
temp_mean = df.select(F.mean(F.col("temperature"))).first()[0]  # Average temperature
wind_median = df.select(F.median(F.col("windspeed"))).first()[0] # Meddle Value
event_mode = df.select(F.mode(F.col("event"))).first()[0] # Most Common Value


temp_mean, wind_median, event_mode


(33.22, 8.1, 'Sunny')

In [0]:
df.na.fill(
    {
        "temperature": temp_mean,
        "windspeed": wind_median,
        "event": event_mode,
    }
).show()

+---------+-----------+---------+------+
|      day|temperature|windspeed| event|
+---------+-----------+---------+------+
| 1/1/2017|       32.0|      6.0|  Rain|
| 1/4/2017|      33.22|      9.0| Sunny|
| 1/5/2017|       28.0|      8.1|  Snow|
| 1/6/2017|      33.22|      7.0| Sunny|
| 1/7/2017|       32.0|      8.1|  Rain|
| 1/8/2017|      33.22|      8.1| Sunny|
| 1/9/2017|      33.22|      8.1| Sunny|
|1/10/2017|       34.1|      8.1|Cloudy|
|1/11/2017|       40.0|     12.0| Sunny|
+---------+-----------+---------+------+

